# Notebook 02 — Génération des Q&R (3 datasets)

**Inputs** :
- `BASE_PATH/data/raw/wikipedia_technique.json`
- `BASE_PATH/data/raw/arxiv.json`
- `BASE_PATH/data/raw/lemonde.json`

**Outputs** (splits figés) :

| Split | Wikipedia (technique) | Arxiv (multisauts) | Le Monde (temporel) | Total |
|-------|-----------------------|--------------------|----------------------|-------|
| train | 100 | 50 simples + 50 complexes | 100 | **300** |
| test  |  40 | 20 simples + 20 complexes |  40 | **120** |

**Runtime** : CPU — ~10 min avec Anthropic claude-3-5-haiku

## 0. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Installation des dépendances

In [ ]:
# Installation du SDK Groq + json-repair pour tolérer le JSON malformé du LLM
!pip install -q groq json-repair

## 2. Imports, configuration et clé API

In [ ]:
# Imports des bibliothèques nécessaires
import os
import json
import time
import random
import getpass
from groq import Groq
from tqdm.notebook import tqdm

# Chemins des répertoires
RAW_PATH       = os.path.join(BASE_PATH, 'data', 'raw')
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

print(f"Répertoire source  : {RAW_PATH}")
print(f"Répertoire cible   : {PROCESSED_PATH}")

In [ ]:
# Saisie sécurisée de la clé Groq Developer (https://console.groq.com → API Keys)
api_key      = getpass.getpass("Entre ta clé Groq API : ")
groq_client  = Groq(api_key=api_key)
print("Client Groq initialisé (plan Developer — pas de rate limit pratique).")

## 3. Chargement des données brutes depuis Drive

In [ ]:
def load_json(path, label=''):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Chargé ({label}) : {len(data)} docs")
        return data
    except FileNotFoundError:
        print(f"  [ERROR] Introuvable : {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON invalide : {e}")
        return []

print("Chargement des 3 datasets bruts...")
wiki_docs    = load_json(os.path.join(RAW_PATH, 'wikipedia_technique.json'), 'Wikipedia technique')
arxiv_docs   = load_json(os.path.join(RAW_PATH, 'arxiv.json'),               'Arxiv multisauts')
lemonde_docs = load_json(os.path.join(RAW_PATH, 'lemonde.json'),             'Le Monde temporel')
print(f"\nTotal : {len(wiki_docs)+len(arxiv_docs)+len(lemonde_docs)} documents")

## 4. Génération des paires Q&R via Anthropic API

In [ ]:
# ── Prompt standard (Wikipedia + Le Monde) ───────────────────────────────────
STANDARD_PROMPT = """Tu es un expert. À partir du texte ci-dessous, génère EXACTEMENT 5 paires question-réponse EN FRANÇAIS.

Génère dans cet ordre :
1. Question FACTUELLE (fait précis, date, chiffre, entité nommée)
2. Question FACTUELLE (idem)
3. Question de SYNTHÈSE (compare ou résume plusieurs concepts)
4. Question de SYNTHÈSE (idem)
5. Question de COMPRÉHENSION (causalité, implication, pourquoi/comment)

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du texte (20-150 mots) qui contient/justifie la réponse.
INTERDIT d'écrire "Texte source", "référence au texte" ou le titre seul.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"..."}}
Types autorisés : "factuel", "synthese", "comprehension"

Exemple :
[{{"question":"En quelle année X a été fondé ?","answer":"2020","context":"X a été fondé en 2020 par des chercheurs issus de Google Brain, avec pour objectif de...","type":"factuel"}}]

Texte source :
{content}"""

# ── Prompt Arxiv simples (3 questions par papier) ────────────────────────────
ARXIV_SIMPLE_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique, génère EXACTEMENT 3 questions factuelles simples EN FRANÇAIS.

Chaque question porte sur UN SEUL fait (méthode utilisée, métrique obtenue, dataset employé, contribution principale).
Chaque réponse tient en 1-2 phrases courtes.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du résumé (20-100 mots) qui contient la réponse.
INTERDIT d'écrire "Texte source" ou similaire.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"simple"}}

Exemple :
[{{"question":"Quel dataset est utilisé pour l'évaluation ?","answer":"MMLU","context":"We evaluate our model on the MMLU benchmark, achieving state-of-the-art performance across 57 tasks.","type":"simple"}}]

Résumé :
{content}"""

# ── Prompt Arxiv complexes / multi-sauts (2 questions par papier) ─────────────
ARXIV_COMPLEX_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique, génère EXACTEMENT 2 questions complexes EN FRANÇAIS.

Ces questions nécessitent de RELIER PLUSIEURS INFORMATIONS du texte pour répondre (multi-sauts de raisonnement).
Par exemple : "Pourquoi la méthode X obtient-elle de meilleurs résultats que Y sur Z ?"
Chaque réponse fait 3-4 phrases et synthétise plusieurs éléments du résumé.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT 1-2 extraits du résumé (30-200 mots) qui, ensemble, permettent de répondre.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"complexe"}}

Résumé :
{content}"""
GROQ_MODEL  = "llama-3.1-8b-instant"
THROTTLE_S  = 0.5    # Groq Developer — pas de rate limit pratique
MAX_CONTENT = 3000   # chars envoyés au LLM

In [ ]:
import re as _re
from json_repair import repair_json

VALID_TYPES = {"factuel", "synthese", "comprehension", "simple", "complexe"}

def _extract_and_repair(raw_text):
    """Extrait le JSON d'une réponse LLM et le répare si nécessaire."""
    # 1. Cherche un tableau JSON entre crochets
    match = _re.search(r'(\[.*\])', raw_text, _re.DOTALL)
    if match:
        candidate = match.group(1)
    else:
        candidate = raw_text.strip()
    # 2. Tente de réparer le JSON malformé
    return repair_json(candidate, return_objects=False)

def _call_groq(prompt, retries=4):
    """Appelle Groq et retourne le texte brut de la réponse."""
    for attempt in range(retries):
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
            return resp.choices[0].message.content
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate_limit' in err.lower():
                wait = 5 * (2 ** attempt)  # 5s, 10s, 20s
                print(f"    [QUOTA] Attente {wait}s...")
                time.sleep(wait)
            else:
                print(f"    [ERROR] {err[:80]}")
                time.sleep(1)
    return ""

def _parse_pairs(raw_text, document, forced_type=None):
    """Extrait et valide les paires Q&R depuis la réponse LLM."""
    content = document.get('content', '')
    json_str = _extract_and_repair(raw_text)
    pairs = json.loads(json_str)
    if not isinstance(pairs, list):
        return []
    validated = []
    for pair in pairs:
        if not (isinstance(pair, dict) and pair.get('question') and pair.get('answer')):
            continue
        q_type = forced_type or str(pair.get('type', 'factuel')).lower().strip()
        if q_type not in VALID_TYPES:
            q_type = forced_type or 'factuel'
        ctx = str(pair.get('context', '')).strip()
        if len(ctx) < 30:
            ctx = content[:400]
        validated.append({
            "question":      str(pair['question']).strip(),
            "answer":        str(pair['answer']).strip(),
            "context":       ctx,
            "source_id":     document.get('id', ''),
            "source":        document.get('source', ''),
            "langue":        "fr",
            "title":         document.get('title', ''),
            "date":          document.get('date', ''),
            "dataset_type":  document.get('dataset_type', ''),
            "question_type": q_type,
        })
    return validated


def generate_standard_qa(document, retries=4):
    """Génère 5 paires standard (Wikipedia / Le Monde)."""
    content = document.get('content', '')
    if len(content) < 100:
        return []
    prompt = STANDARD_PROMPT.format(content=content[:MAX_CONTENT])
    for attempt in range(retries):
        try:
                                    raw = _call_groq(prompt)
            if not raw:
                continue
            pairs = _parse_pairs(raw, document)
            if pairs:
                return pairs
        except (json.JSONDecodeError, ValueError) as e:
            print(f"    [WARN] {attempt+1}/{retries} JSON invalide '{document.get('title','')[:30]}': {str(e)[:50]}")
            time.sleep(2 ** attempt)
    return []


def generate_arxiv_qa(document, retries=4):
    """Génère 3 questions simples + 2 questions complexes pour un papier Arxiv."""
    content = document.get('content', '')
    if len(content) < 100:
        return []
    all_pairs = []
    for prompt_template, forced_type, n_expected in [
        (ARXIV_SIMPLE_PROMPT,  'simple',   3),
        (ARXIV_COMPLEX_PROMPT, 'complexe', 2),
    ]:
        prompt = prompt_template.format(content=content[:MAX_CONTENT])
        for attempt in range(retries):
            try:
                                                raw = _call_groq(prompt)
                if not raw:
                    break
                pairs = _parse_pairs(raw, document, forced_type=forced_type)
                if pairs:
                    all_pairs.extend(pairs[:n_expected])
                    break
            except (json.JSONDecodeError, ValueError) as e:
                print(f"    [WARN] {attempt+1}/{retries} Arxiv {forced_type} '{document.get('title','')[:30]}': {str(e)[:50]}")
                time.sleep(2 ** attempt)
        time.sleep(THROTTLE_S)
    return all_pairs

In [ ]:
print("=" * 60)
print("Génération des Q&R — 3 datasets")
print("=" * 60)

# ── Wikipedia (standard) ─────────────────────────────────────────────────────
print("\n[1/3] Wikipedia technique...")
wiki_qa = []
for doc in tqdm(wiki_docs, desc="Wikipedia Q&R"):
    pairs = generate_standard_qa(doc)
    for i, p in enumerate(pairs):
        p['pair_id'] = f"wiki_qa_{len(wiki_qa):04d}"
        wiki_qa.append(p)
    time.sleep(THROTTLE_S)
print(f"  => {len(wiki_qa)} paires générées depuis {len(wiki_docs)} articles")

# ── Arxiv (simple + complexe) ────────────────────────────────────────────────
print("\n[2/3] Arxiv multisauts...")
arxiv_qa = []
for doc in tqdm(arxiv_docs, desc="Arxiv Q&R"):
    pairs = generate_arxiv_qa(doc)
    for p in pairs:
        p['pair_id'] = f"arxiv_qa_{len(arxiv_qa):04d}"
        arxiv_qa.append(p)
# (throttle already inside generate_arxiv_qa)
arxiv_simple  = [p for p in arxiv_qa if p['question_type'] == 'simple']
arxiv_complex = [p for p in arxiv_qa if p['question_type'] == 'complexe']
print(f"  => {len(arxiv_qa)} paires  ({len(arxiv_simple)} simples + {len(arxiv_complex)} complexes)")

# ── Le Monde (standard) ──────────────────────────────────────────────────────
print("\n[3/3] Le Monde temporel...")
lemonde_qa = []
for doc in tqdm(lemonde_docs, desc="Le Monde Q&R"):
    pairs = generate_standard_qa(doc)
    for p in pairs:
        p['pair_id'] = f"lemonde_qa_{len(lemonde_qa):04d}"
        lemonde_qa.append(p)
    time.sleep(THROTTLE_S)
print(f"  => {len(lemonde_qa)} paires générées depuis {len(lemonde_docs)} articles")

## 5. Mélange et division train / test

In [ ]:
import random
random.seed(42)

def exact_split(pairs, n_train, n_test, label=''):
    """Sélectionne exactement n_train + n_test paires (shuffle reproductible)."""
    shuffled = pairs[:]
    random.shuffle(shuffled)
    train = shuffled[:n_train]
    test  = shuffled[n_train:n_train + n_test]
    total = len(pairs)
    if total < n_train + n_test:
        print(f"  [WARN] {label}: seulement {total} paires dispo pour {n_train}+{n_test} demandées")
    return train, test

# ── Wikipedia : 100 train + 40 test ─────────────────────────────────────────
wiki_train, wiki_test = exact_split(wiki_qa, 100, 40, 'Wikipedia')

# ── Arxiv : équilibre simple/complexe ────────────────────────────────────────
arxiv_s_train, arxiv_s_test = exact_split(arxiv_simple,  50, 20, 'Arxiv simple')
arxiv_c_train, arxiv_c_test = exact_split(arxiv_complex, 50, 20, 'Arxiv complexe')
arxiv_train = arxiv_s_train + arxiv_c_train
arxiv_test  = arxiv_s_test  + arxiv_c_test

# ── Le Monde : 100 train + 40 test ───────────────────────────────────────────
lemonde_train, lemonde_test = exact_split(lemonde_qa, 100, 40, 'Le Monde')

# ── Assemblage final ─────────────────────────────────────────────────────────
train_data = wiki_train + arxiv_train + lemonde_train
test_data  = wiki_test  + arxiv_test  + lemonde_test
random.shuffle(train_data)
random.shuffle(test_data)

# Re-indexation des pair_id
for i, p in enumerate(train_data): p['pair_id'] = f"train_{i:04d}"
for i, p in enumerate(test_data):  p['pair_id'] = f"test_{i:04d}"

import collections
print("\n" + "=" * 55)
print("RÉPARTITION FINALE DU DATASET")
print("=" * 55)
for split_name, split in [("TRAIN (300)", train_data), ("TEST  (120)", test_data)]:
    print(f"\n  {split_name}")
    by_ds = collections.Counter(p['dataset_type']   for p in split)
    by_qt = collections.Counter(p['question_type']  for p in split)
    print(f"    Par source     : {dict(by_ds)}")
    print(f"    Par type Q     : {dict(by_qt)}")
print(f"\n  Totaux : {len(train_data)} train + {len(test_data)} test = {len(train_data)+len(test_data)} paires")

## 6. Statistiques du dataset

In [ ]:
# Statistiques détaillées du dataset train + test
import statistics, re

def field_stats(pairs, field):
    counts = [len(str(p.get(field, '')).split()) for p in pairs]
    if not counts:
        return None
    return {"min": min(counts), "mean": round(statistics.mean(counts),1),
            "median": statistics.median(counts), "max": max(counts), "total": sum(counts)}

def print_split_stats(split_name, pairs):
    print(f"\n  ── {split_name} ({len(pairs)} paires) ──")
    print(f"  {'Champ':<12} {'Min':>5} {'Moy':>7} {'Médiane':>8} {'Max':>5} {'Total mots':>12}")
    print(f"  {'─'*12} {'─'*5} {'─'*7} {'─'*8} {'─'*5} {'─'*12}")
    for field, label in [("question","Question"), ("answer","Réponse"), ("context","Contexte")]:
        s = field_stats(pairs, field)
        if s:
            print(f"  {label:<12} {s['min']:>5} {s['mean']:>7} {s['median']:>8.0f} {s['max']:>5} {s['total']:>12,}")

    # Types de questions
    type_dist = {}
    for p in pairs:
        t = p.get('question_type', 'inconnu')
        type_dist[t] = type_dist.get(t, 0) + 1
    print(f"\n  Types de questions :")
    for t, cnt in sorted(type_dist.items(), key=lambda x: -x[1]):
        bar = "█" * (cnt // max(1, len(pairs) // 30))
        print(f"    {t:<16} : {cnt:>4} ({100*cnt/len(pairs):.1f}%)  {bar}")

    # Strates temporelles
    rc_dist = {}
    for p in pairs:
        rc = p.get('recency_category', 'inconnu')
        rc_dist[rc] = rc_dist.get(rc, 0) + 1
    print(f"\n  Strates temporelles :")
    for cat in ["récent", "intermédiaire", "fondamental", "inconnu"]:
        cnt = rc_dist.get(cat, 0)
        if cnt > 0:
            print(f"    {cat:<16} : {cnt:>4} ({100*cnt/len(pairs):.1f}%)")

    # Sources
    src_dist = {}
    for p in pairs:
        src_dist[p.get('source','?')] = src_dist.get(p.get('source','?'), 0) + 1
    print(f"\n  Sources :")
    for src, cnt in sorted(src_dist.items(), key=lambda x: -x[1]):
        print(f"    {src:<20} : {cnt:>4} ({100*cnt/len(pairs):.1f}%)")

print("=" * 65)
print("STATISTIQUES DU DATASET (langue : français)")
print("=" * 65)
print_split_stats("TRAIN", train_data)
print_split_stats("TEST ", test_data)

all_qa_pairs = wiki_qa + arxiv_qa + lemonde_qa
all_text = " ".join(p.get('question','') + " " + p.get('answer','') for p in all_qa_pairs)
vocab = set(re.sub(r'[^a-zàâéèêëîïôùûüç\s]', '', all_text.lower()).split())
print(f"\n  Vocabulaire unique (approx.) : {len(vocab):,} tokens")
print(f"  Total paires générées        : {len(all_qa_pairs)}")
print(f"  Paires conservées            : {len(train_data)+len(test_data)} (train {len(train_data)} + test {len(test_data)})")
print("=" * 65)
print("\n✔ Notebook 02 terminé. Lancez 03_baseline_rag.ipynb pour la suite.")

## 6. Sauvegarde sur Drive

In [ ]:
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

train_path = os.path.join(PROCESSED_PATH, 'train.json')
test_path  = os.path.join(PROCESSED_PATH, 'test.json')

for data, path, label in [(train_data, train_path, 'train'), (test_data, test_path, 'test')]:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"{label}.json : {len(data)} paires  ({os.path.getsize(path)/1024:.1f} Ko)  → {path}")

## 7. Résumé final

In [ ]:
import os
train_path = os.path.join(BASE_PATH, 'data', 'processed', 'train.json')
test_path  = os.path.join(BASE_PATH, 'data', 'processed', 'test.json')

print("=" * 60)
print("RÉSUMÉ FINAL — Notebook 02 Dataset Builder")
print("=" * 60)
print(f"\n  Données brutes chargées :")
print(f"    Wikipedia  : {len(wiki_docs)} articles")
print(f"    Arxiv      : {len(arxiv_docs)} papiers")
print(f"    Le Monde   : {len(lemonde_docs)} articles")
print(f"\n  Paires Q&R générées :")
print(f"    Wikipedia  : {len(wiki_qa)} paires")
print(f"    Arxiv      : {len(arxiv_qa)} paires  ({len(arxiv_simple)} simples + {len(arxiv_complex)} complexes)")
print(f"    Le Monde   : {len(lemonde_qa)} paires")
print(f"    Total      : {len(wiki_qa)+len(arxiv_qa)+len(lemonde_qa)} paires générées")
print(f"\n  Splits sauvegardés :")
print(f"    train.json : {len(train_data)} paires  → {train_path}")
print(f"    test.json  : {len(test_data)} paires  → {test_path}")
print(f"    Total      : {len(train_data)+len(test_data)} paires conservées")
print(f"\n  Fichiers sur Drive :")
for p in [train_path, test_path]:
    if os.path.exists(p):
        print(f"    ✓ {os.path.basename(p)}  ({os.path.getsize(p)/1024:.1f} Ko)")
    else:
        print(f"    ✗ {os.path.basename(p)}  INTROUVABLE")
print("\n→ Lancez 03_baseline_rag.ipynb pour l'étape suivante.")
print("=" * 60)
